# SelvaSonic — Comparación de Embeddings: Baseline vs Attention (Semana 5.5)

## Objetivo

El cronograma de Semana 5 pide responder visualmente:

> **¿Se separan las especies en el espacio de embeddings? ¿El attention mejora esa separación?**

Como tenemos dos modelos entrenados bajo condiciones idénticas, podemos comparar **directamente** las representaciones internas que ambos aprendieron. Esto es más valioso que analizar solo uno: nos permite ver **el efecto del attention sobre la geometría del espacio de features**.

## ¿Qué es un embedding y dónde lo sacamos?

Un **embedding** es la representación interna que el modelo construye del audio antes de la capa final de clasificación. Si el modelo aprendió bien, espectrogramas de la misma especie deberían quedar **cerca** en este espacio, y los de especies distintas **lejos**.

Para ambos modelos, el embedding natural es el vector de **256 dimensiones** justo antes del clasificador final:

- **Baseline (`SelvaSonicCNN`)**: salida del `global_pool` (AdaptiveAvgPool 1x1 sobre el feature map de 256 canales).
- **Attention (`SelvaSonicCNNAttention`)**: media sobre los tokens después de la capa de attention + LayerNorm.

Ambos tienen la misma dimensionalidad (256), lo que hace la comparación geométricamente justa.

## Estructura del notebook

| Sección | Contenido |
|---|---|
| 1. Setup | Cargar ambos modelos, reconstruir test set |
| 2. Extracción de embeddings | 256-d para baseline y attention |
| 3. t-SNE lado a lado | Comparación visual con vecindad local |
| 4. UMAP lado a lado | Comparación con estructura global |
| 5. Silhouette scores | Métrica cuantitativa de separación |
| 6. Análisis del género Crypturellus | ¿Mejoró la separación intra-género? |
| 7. Conclusiones | Hallazgos para el reporte |

## Sección 1 — Setup

Asegúrate de tener `umap-learn` instalado:

```
pip install umap-learn
```

(t-SNE viene con scikit-learn que ya tienes).

In [ ]:
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if (PROJECT_ROOT / 'src').exists():
    sys.path.insert(0, str(PROJECT_ROOT))
elif (PROJECT_ROOT.parent / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score, silhouette_samples

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print('AVISO: umap-learn no instalado. Instala con: pip install umap-learn')

from src.dataset import create_dataloaders
from src.model import SelvaSonicCNN, SelvaSonicCNNAttention

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
COLOR_DARK = '#2D3436'

BASELINE_DIR = PROJECT_ROOT / 'results' / 'runs' / 'baseline_S3_v2_20260527_0118'
ATTENTION_DIR = PROJECT_ROOT / 'results' / 'runs' / 'attention_S4_v1_20260601_0334'
OUT_DIR = PROJECT_ROOT / 'results' / 'comparacion_baseline_vs_attention'
OUT_DIR.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR = PROJECT_ROOT / 'data' / 'raw'

assert (BASELINE_DIR / 'best.pth').exists(), 'Falta best.pth del baseline'
assert (ATTENTION_DIR / 'best.pth').exists(), 'Falta best.pth del attention'
print(f'Device: {device}')
print(f'UMAP disponible: {UMAP_AVAILABLE}')

In [ ]:
# Reconstruir test set con misma semilla
_, _, test_loader, label_map = create_dataloaders(
    raw_data_dir=str(RAW_DATA_DIR),
    batch_size=32, num_workers=0,
    train_ratio=0.70, val_ratio=0.15, test_ratio=0.15,
    random_state=42, verbose=False,
)
NUM_CLASSES = len(label_map)
idx_to_name = {v: k for k, v in label_map.items()}
class_names = [idx_to_name[i] for i in range(NUM_CLASSES)]

# Cargar baseline
ckpt_b = torch.load(BASELINE_DIR / 'best.pth', map_location=device)
model_baseline = SelvaSonicCNN(num_classes=NUM_CLASSES).to(device)
model_baseline.load_state_dict(ckpt_b['model_state_dict'])
model_baseline.eval()

# Cargar attention
ckpt_a = torch.load(ATTENTION_DIR / 'best.pth', map_location=device)
model_attention = SelvaSonicCNNAttention(num_classes=NUM_CLASSES).to(device)
model_attention.load_state_dict(ckpt_a['model_state_dict'])
model_attention.eval()

print(f'Baseline:  epoca {ckpt_b["epoch"]+1}, val_acc={ckpt_b["best_val_acc"]:.4f}')
print(f'Attention: epoca {ckpt_a["epoch"]+1}, val_acc={ckpt_a["best_val_acc"]:.4f}')

## Sección 2 — Extracción de embeddings

### ¿Cómo extraemos los embeddings?

Usamos **PyTorch forward hooks**: funciones que se ejecutan automáticamente cuando un módulo procesa datos. Aquí los usamos para capturar la salida intermedia del modelo (justo antes del clasificador) sin tener que modificar las clases.

Para cada modelo, el hook se aplica al módulo que produce el vector de 256 dimensiones:
- **Baseline**: el `global_pool` (su output es `(B, 256, 1, 1)`).
- **Attention**: la `layer_norm` después del attention (su output es `(B, 112, 256)`; aplicamos `mean(dim=1)` para obtener `(B, 256)`).

Esto garantiza que ambos embeddings sean del mismo tamaño y comparables.

In [ ]:
def extraer_embeddings_baseline(model, loader):
    """Extrae embeddings de SelvaSonicCNN (output del global_pool)."""
    buffer, labels = [], []
    def hook(module, inp, out):
        # out: (B, 256, 1, 1) -> (B, 256)
        buffer.append(out.detach().cpu().squeeze(-1).squeeze(-1))
    
    handle = model.global_pool.register_forward_hook(hook)
    with torch.no_grad():
        for x, y in loader:
            _ = model(x.to(device))
            labels.append(y.numpy())
    handle.remove()
    return torch.cat(buffer).numpy(), np.concatenate(labels)


def extraer_embeddings_attention(model, loader):
    """Extrae embeddings de SelvaSonicCNNAttention (mean sobre tokens despues de attention+LN)."""
    buffer, labels = [], []
    def hook(module, inp, out):
        # out: (B, 112, 256) -> mean -> (B, 256)
        buffer.append(out.detach().cpu().mean(dim=1))
    
    handle = model.layer_norm.register_forward_hook(hook)
    with torch.no_grad():
        for x, y in loader:
            _ = model(x.to(device))
            labels.append(y.numpy())
    handle.remove()
    return torch.cat(buffer).numpy(), np.concatenate(labels)


print('Extrayendo embeddings del baseline...')
emb_baseline, labels = extraer_embeddings_baseline(model_baseline, test_loader)
print(f'  Shape: {emb_baseline.shape}')

print('Extrayendo embeddings del attention...')
emb_attention, labels2 = extraer_embeddings_attention(model_attention, test_loader)
print(f'  Shape: {emb_attention.shape}')

# Verificar que las etiquetas coinciden (mismo test set, mismo orden)
assert np.array_equal(labels, labels2), 'Las etiquetas no coinciden entre modelos!'
print(f'\nOK Etiquetas coinciden ({len(labels)} clips)')
print(f'Estadisticas baseline:  min={emb_baseline.min():.2f}, max={emb_baseline.max():.2f}, std={emb_baseline.std():.2f}')
print(f'Estadisticas attention: min={emb_attention.min():.2f}, max={emb_attention.max():.2f}, std={emb_attention.std():.2f}')

## Sección 3 — Visualización con t-SNE lado a lado

t-SNE preserva **vecindades locales**: si dos puntos están cerca en 256-d, lo estarán en 2-d. Es bueno para ver clusters compactos. Usamos los mismos hiperparámetros para ambos modelos (`perplexity=30`, `random_state=42`) para que la comparación sea justa.

In [ ]:
print('Ejecutando t-SNE para baseline... (1-2 min)')
tsne_baseline = TSNE(n_components=2, perplexity=30, init='pca', random_state=42, n_iter=1000).fit_transform(emb_baseline)

print('Ejecutando t-SNE para attention... (1-2 min)')
tsne_attention = TSNE(n_components=2, perplexity=30, init='pca', random_state=42, n_iter=1000).fit_transform(emb_attention)

print('Listo.')

In [ ]:
# Visualizacion lado a lado
fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.patch.set_facecolor('#FAFAFA')
cmap = plt.cm.tab20

for ax, emb_2d, titulo in [
    (axes[0], tsne_baseline, 'BASELINE (SelvaSonicCNN)'),
    (axes[1], tsne_attention, 'ATTENTION (SelvaSonicCNNAttention)'),
]:
    for i in range(NUM_CLASSES):
        mask = labels == i
        ax.scatter(
            emb_2d[mask, 0], emb_2d[mask, 1],
            c=[cmap(i / NUM_CLASSES)], label=class_names[i],
            alpha=0.65, s=24, edgecolors='white', linewidths=0.3,
        )
    ax.set_title(f't-SNE — {titulo}', fontsize=12, color=COLOR_DARK)
    ax.set_xlabel('dim 1'); ax.set_ylabel('dim 2')
    ax.grid(alpha=0.2)

# Leyenda compartida fuera
axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, framealpha=0.95)
plt.suptitle('Comparación de embeddings t-SNE: Baseline vs Attention',
             fontsize=14, color=COLOR_DARK, y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'embeddings_tsne_comparacion.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
plt.show()
print(f'Guardada: embeddings_tsne_comparacion.png')

### Cómo leer estas gráficas

Visualmente, busca:

- **Clusters más compactos en attention** = el modelo de attention agrupa mejor los clips de la misma especie.
- **Mayor separación entre clusters** = especies más fácilmente distinguibles.
- **Menos puntos perdidos en "tierra de nadie"** = menos clips ambiguos.

Si el attention mejora la geometría del espacio, la gráfica de la derecha debería verse "más limpia" que la de la izquierda.

## Sección 4 — Visualización con UMAP lado a lado

UMAP es complementario a t-SNE: preserva mejor la **estructura global** (las distancias entre clusters son más interpretables). Hacer ambos análisis y ver si coinciden es una práctica estándar en ML aplicado.

In [ ]:
if UMAP_AVAILABLE:
    print('Ejecutando UMAP para baseline... (~30s)')
    umap_baseline = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(emb_baseline)
    
    print('Ejecutando UMAP para attention... (~30s)')
    umap_attention = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(emb_attention)
    
    print('Listo.')
else:
    print('UMAP no disponible. Saltando sección.')

In [ ]:
if UMAP_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(20, 9))
    fig.patch.set_facecolor('#FAFAFA')
    
    for ax, emb_2d, titulo in [
        (axes[0], umap_baseline, 'BASELINE'),
        (axes[1], umap_attention, 'ATTENTION'),
    ]:
        for i in range(NUM_CLASSES):
            mask = labels == i
            ax.scatter(
                emb_2d[mask, 0], emb_2d[mask, 1],
                c=[cmap(i / NUM_CLASSES)], label=class_names[i],
                alpha=0.65, s=24, edgecolors='white', linewidths=0.3,
            )
        ax.set_title(f'UMAP — {titulo}', fontsize=12, color=COLOR_DARK)
        ax.set_xlabel('dim 1'); ax.set_ylabel('dim 2')
        ax.grid(alpha=0.2)
    
    axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, framealpha=0.95)
    plt.suptitle('Comparación de embeddings UMAP: Baseline vs Attention',
                 fontsize=14, color=COLOR_DARK, y=1.02)
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'embeddings_umap_comparacion.png', dpi=150, bbox_inches='tight', facecolor='#FAFAFA')
    plt.show()
    print(f'Guardada: embeddings_umap_comparacion.png')

## Sección 5 — Silhouette scores: medir la separación cuantitativamente

Las gráficas son intuitivas pero subjetivas. Para el reporte necesitamos un **número**.

El **silhouette score** mide qué tan bien definido está cada cluster:

$$ s(i) = \frac{b(i) - a(i)}{\max\{a(i), b(i)\}} $$

donde $a(i)$ es la distancia media de $i$ a su propio cluster (cohesión) y $b(i)$ es la distancia al cluster vecino más cercano (separación). Rango **[-1, 1]**:
- **+1**: punto muy bien clasificado.
- **0**: punto en la frontera.
- **-1**: punto mal asignado.

**Calculamos sobre el espacio original de 256-d**, no sobre la proyección 2-d (las proyecciones distorsionan distancias).

In [ ]:
sil_b = silhouette_score(emb_baseline, labels, metric='euclidean')
sil_a = silhouette_score(emb_attention, labels, metric='euclidean')

print(f'SILHOUETTE SCORE GLOBAL:\n')
print(f'  Baseline:  {sil_b:+.4f}')
print(f'  Attention: {sil_a:+.4f}')
print(f'  Delta:     {sil_a - sil_b:+.4f}')
print(f'\nInterpretacion: {"El attention MEJORA la separacion de clusters." if sil_a > sil_b else "El attention NO mejora la separacion."}')

# Por clase
sil_samples_b = silhouette_samples(emb_baseline, labels)
sil_samples_a = silhouette_samples(emb_attention, labels)

print('\nSILHOUETTE POR CLASE:\n')
print(f'{"Clase":<25} {"Baseline":>10} {"Attention":>10} {"Delta":>10}')
print('-' * 60)
deltas = []
for i in range(NUM_CLASSES):
    mask = labels == i
    if mask.sum() > 0:
        sb = sil_samples_b[mask].mean()
        sa = sil_samples_a[mask].mean()
        deltas.append((class_names[i], sb, sa, sa - sb))

# Ordenar por mayor mejora
deltas.sort(key=lambda x: -x[3])
for name, sb, sa, d in deltas:
    flag = ' MEJORA' if d > 0.02 else (' empeora' if d < -0.02 else '')
    print(f'{name:<25} {sb:>+10.3f} {sa:>+10.3f} {d:>+10.3f}{flag}')

## Sección 6 — Foco en Crypturellus: ¿el attention separó las dos especies del mismo género?

El análisis de errores del baseline (notebook 04) detectó confusiones fuertes entre las dos especies del género *Crypturellus*. La hipótesis era que el attention podría aprender a distinguirlas mejor al enfocarse en regiones discriminativas del espectrograma.

Verifiquémoslo geométricamente: ¿los embeddings de *Crypturellus_cinereus* y *Crypturellus_undulatus* se separaron más con el attention?

In [ ]:
from scipy.spatial.distance import cdist

# Indices de las dos especies Crypturellus
if 'Crypturellus_cinereus' in label_map and 'Crypturellus_undulatus' in label_map:
    idx_cin = label_map['Crypturellus_cinereus']
    idx_und = label_map['Crypturellus_undulatus']
    
    mask_cin = labels == idx_cin
    mask_und = labels == idx_und
    
    print(f'Clips de Crypturellus_cinereus:  {mask_cin.sum()}')
    print(f'Clips de Crypturellus_undulatus: {mask_und.sum()}')
    
    # Distancia media entre clusters en ambos espacios
    dist_baseline = cdist(emb_baseline[mask_cin], emb_baseline[mask_und]).mean()
    dist_attention = cdist(emb_attention[mask_cin], emb_attention[mask_und]).mean()
    
    # Normalizar por la dispersion intra-cluster para que sea comparable
    intra_cin_b = cdist(emb_baseline[mask_cin], emb_baseline[mask_cin]).mean()
    intra_und_b = cdist(emb_baseline[mask_und], emb_baseline[mask_und]).mean()
    intra_cin_a = cdist(emb_attention[mask_cin], emb_attention[mask_cin]).mean()
    intra_und_a = cdist(emb_attention[mask_und], emb_attention[mask_und]).mean()
    
    separabilidad_b = dist_baseline / ((intra_cin_b + intra_und_b) / 2)
    separabilidad_a = dist_attention / ((intra_cin_a + intra_und_a) / 2)
    
    print(f'\nDistancia media inter-cluster:')
    print(f'  Baseline:  {dist_baseline:.3f}')
    print(f'  Attention: {dist_attention:.3f}')
    print(f'\nRatio de separabilidad (inter / intra):')
    print(f'  Baseline:  {separabilidad_b:.3f}')
    print(f'  Attention: {separabilidad_a:.3f}')
    print(f'  Mejora:    {(separabilidad_a/separabilidad_b - 1)*100:+.1f}%')
    
    if separabilidad_a > separabilidad_b:
        print(f'\n>>> CONFIRMADO: el attention separa MEJOR las dos Crypturellus en el espacio de embeddings.')
    else:
        print(f'\n>>> El attention NO logro separar mejor las Crypturellus geometricamente.')
else:
    print('No se encontraron ambas especies Crypturellus en el label_map.')

## Sección 7 — Resumen para el reporte

In [ ]:
# Top 3 clases que mas mejoraron en silhouette
top_mejoras = sorted(deltas, key=lambda x: -x[3])[:3]
top_caidas = sorted(deltas, key=lambda x: x[3])[:3]

resumen = f"""
{'=' * 75}
COMPARACION DE EMBEDDINGS — BASELINE vs ATTENTION
{'=' * 75}

METODOLOGIA
  Embeddings de 256-d extraidos del test set comun a ambos modelos.
  - Baseline:  output del global_pool en SelvaSonicCNN.
  - Attention: mean sobre tokens despues de attention+LN en SelvaSonicCNNAttention.
  Visualizacion: t-SNE (perplexity=30) y UMAP (n_neighbors=15).
  Metrica: silhouette score sobre el espacio original 256-d.

SILHOUETTE SCORE GLOBAL
  Baseline:  {sil_b:+.4f}
  Attention: {sil_a:+.4f}
  Delta:     {sil_a - sil_b:+.4f}  ({(sil_a/sil_b - 1)*100:+.1f}% relativo)

CLASES QUE MAS GANARON EN SEPARACION
{chr(10).join(f"  {name:<25} delta = {d:+.3f}" for name, _, _, d in top_mejoras)}

CLASES QUE PERDIERON EN SEPARACION (si las hay)
{chr(10).join(f"  {name:<25} delta = {d:+.3f}" for name, _, _, d in top_caidas if d < 0)}

HALLAZGOS
  1. El silhouette score {'mejoro' if sil_a > sil_b else 'no mejoro'} globalmente con el attention.
     Esto {'confirma' if sil_a > sil_b else 'matiza'} que el modulo de atencion construye
     un espacio de representaciones {'mas' if sil_a > sil_b else 'no necesariamente mas'} separable.
  2. La mejora del macro F1 (+9.0%) {'se correlaciona' if sil_a > sil_b else 'no se correlaciona'} con la mejora
     geometrica de los embeddings.
  3. Visualmente (t-SNE y UMAP), las dos representaciones muestran patrones
     distintos: el attention tiende a producir clusters {'mas compactos' if sil_a > sil_b else 'similares'}.

ARTEFACTOS GENERADOS
  - results/comparacion_baseline_vs_attention/embeddings_tsne_comparacion.png
  - results/comparacion_baseline_vs_attention/embeddings_umap_comparacion.png
  - results/comparacion_baseline_vs_attention/embeddings_resumen.txt
  - results/comparacion_baseline_vs_attention/embeddings_data.npz

{'=' * 75}
"""
print(resumen)

with open(OUT_DIR / 'embeddings_resumen.txt', 'w', encoding='utf-8') as f:
    f.write(resumen)

# Guardar embeddings y proyecciones para reuso futuro
save_dict = {
    'emb_baseline_256d': emb_baseline,
    'emb_attention_256d': emb_attention,
    'tsne_baseline': tsne_baseline,
    'tsne_attention': tsne_attention,
    'labels': labels,
    'class_names': np.array(class_names),
    'silhouette_baseline': sil_b,
    'silhouette_attention': sil_a,
}
if UMAP_AVAILABLE:
    save_dict['umap_baseline'] = umap_baseline
    save_dict['umap_attention'] = umap_attention

np.savez(OUT_DIR / 'embeddings_data.npz', **save_dict)
print(f'Datos guardados: embeddings_data.npz')

## Cierre

Este notebook completa **S5.5 del cronograma**: análisis comparativo de embeddings entre baseline y attention. Los resultados aquí, junto con los del notebook 09, son la **evidencia central** de la contribución del módulo de atención al proyecto.

**Para el reporte**, las dos gráficas estrella son:
- `embeddings_tsne_comparacion.png` (visual lado a lado)
- `f1_por_clase_comparacion.png` (del notebook 09)

Ambas cuentan la historia completa en dos imágenes.

**Siguiente paso del cronograma:** S6 — script de inferencia, demo, README, y reporte final.